# Test notebook for MCP tools

It tries every MCP tool used by AACAgent. `get_time` / `get_schedule` hit the
real system clock and the real calendar provider. 

> **Look at `app/README.md` for setting up tools.**

Only MCP tool APIs are used.

The index is:
1. [Setup](#0-setup)
2. [`list_keywords`](#1-list_keywords)
3. [`search_pictograms`](#2-search_pictograms)
4. [`get_pictogram_metadata`](#3-get_pictogram_metadata)
5. [`search_pictograms_by_synset`](#4-search_pictograms_by_synset)
6. [`get_time`](#5-get_time)
7. [`get_schedule`](#6-get_schedule)
8. [Online / offline comparison](#7-online--offline-comparison)
9. [`resolve_concept`](#8-resolve_concept)


## 1. Setup

In [2]:
import sys
from pathlib import Path

HERE = Path('.').resolve()
PROJECT_ROOT = HERE.parent if (HERE / 'tools_test.ipynb').exists() else HERE
SRC = PROJECT_ROOT / 'app' / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print('PROJECT_ROOT:', PROJECT_ROOT)
print('SRC:', SRC)


PROJECT_ROOT: /Users/pelle/Development/GitHub/aac-mcp-agent
SRC: /Users/pelle/Development/GitHub/aac-mcp-agent/app/src


In [3]:
from config import LANG, USE_LOCAL_DATASETS, DATASETS_DIR
from mcp_server.tools.arasaac import (
    list_keywords,
    search_pictograms,
    get_pictogram_metadata,
    search_pictograms_by_synset,
)
from mcp_server.tools.time_tool import get_time
from mcp_server.tools.schedule_tool import get_schedule
import mcp_server.tools.arasaac as _arasaac_mod  # for monkey-patching

print(f'LANG={LANG!r}  USE_LOCAL_DATASETS={USE_LOCAL_DATASETS}')
print(f'DATASETS_DIR exists: {DATASETS_DIR.exists()}')


LANG='en'  USE_LOCAL_DATASETS=True
DATASETS_DIR exists: True


## 1. `list_keywords`

How many keywords there are, and what they look like.

In [4]:
kws = list_keywords(lang=LANG)['keywords']
kw_set = set(kws)

print(f'Total keywords: {len(kws):,}')
print(f'First 10: {kws[:10]}')


[07/19/26 15:00:16] INFO     Loaded 15718 keywords for lang='en'.                               ]8;id=2645328;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/dataset_cache.py\dataset_cache.py]8;;\:]8;id=2645329;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/dataset_cache.py#68\68]8;;\

                    INFO     list_keywords(lang='en'): 15718 keywords from local dataset.            ]8;id=2645336;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=2645337;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#280\280]8;;\

Total keywords: 15,718
First 10: ['!', '"', '#', '$', '%', '(', ')', '*', '+', ',']


In [5]:
# A few known keywords, just to get a feel for what's in there
for k in ['water', 'eat', 'coat', 'bag', 'shoes']:
    print(f'{k!r:>10} in index: {k in kw_set}')

multi = [k for k in kw_set if ' ' in k]
print(f'\nMulti-word keywords: {len(multi):,} ({len(multi)/len(kw_set)*100:.1f}%)')
print('Examples:', multi[:5])


   'water' in index: True
     'eat' in index: True
    'coat' in index: True
     'bag' in index: True
   'shoes' in index: True

Multi-word keywords: 7,871 (50.1%)
Examples: ['picnic table', 'casino chips', 'coffee capsules', 'observation ward', 'medical emergency phone']


## 2. `search_pictograms`

In [6]:
res = search_pictograms(keyword='water', lang=LANG, max_results=5)
pics = res['results']

print(f'search_pictograms("water") --> {len(pics)} results')
for p in pics:
    print(f'  id={p["id"]:<8} kws={[k["keyword"] for k in p.get("keywords", [])[:3]]}')


                    INFO     Loaded keyword_index for lang='en' (15718 entries).                ]8;id=2645343;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/dataset_cache.py\dataset_cache.py]8;;\:]8;id=2645344;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/dataset_cache.py#90\90]8;;\

                    INFO     Loaded 13800 pictograms for lang='en'.                             ]8;id=2645350;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/dataset_cache.py\dataset_cache.py]8;;\:]8;id=2645351;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/dataset_cache.py#79\79]8;;\

                    INFO     search_pictograms('water', lang='en'): 5 results from local dataset.    ]8;id=2645357;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=2645358;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#206\206]8;;\

search_pictograms("water") --> 5 results
  id=22082    kws=['water', 'irrigate']
  id=2248     kws=['water']
  id=24354    kws=['water', 'spray', 'irrigate']
  id=24355    kws=['water', 'irrigate', 'spray']
  id=2816     kws=['water']


In [7]:
# One full result, to see every field that actually comes back
pics[0]


{'id': 22082,
 'image_url': '/api/images/22082',
 'keywords': [{'type': 3,
   'keyword': 'water',
   'plural': None,
   'meaning': 'tr. Sprinkle water on a surface: watering the garden.'},
  {'type': 3, 'keyword': 'irrigate', 'plural': None, 'meaning': None}],
 'categories': ['verb', 'agriculture', 'gardening'],
 'synsets': ['00228662-v'],
 'tags': ['communication',
  'language',
  'verb',
  'work',
  'primary sector',
  'agriculture',
  'gardening'],
 'sex': False,
 'violence': False,
 'schematic': False,
 'aac': False,
 'aac_color': False,
 'skin': True,
 'hair': True,
 'created': '2010-10-03T17:05:18.000Z',
 'last_updated': '2020-11-21T00:09:27.943Z'}

In [8]:
# max_results, multi-word keyword, unknown keyword
print('max_results=2:', len(search_pictograms(keyword='eat', lang=LANG, max_results=2)['results']), 'results')

if 'go out' in kw_set:
    n = len(search_pictograms(keyword='go out', lang=LANG, max_results=5)['results'])
    print(f'"go out" (multi-word): {n} results')

print('unknown keyword:', search_pictograms(keyword='fheoufhueifnwfou', lang=LANG)['results'])


                    INFO     search_pictograms('eat', lang='en'): 2 results from local dataset.      ]8;id=2645363;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=2645364;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#206\206]8;;\

max_results=2: 2 results


                    INFO     search_pictograms('go out', lang='en'): 4 results from local dataset.   ]8;id=2645369;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=2645370;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#206\206]8;;\

"go out" (multi-word): 4 results


                    INFO     search_pictograms('fheoufhueifnwfou', lang='en'): 0 results from local  ]8;id=2645375;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=2645376;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#206\206]8;;\
                             dataset.                                                                              

unknown keyword: []


## 3. `get_pictogram_metadata`

In [9]:
WATER_ID = 2248
meta = get_pictogram_metadata(pictogram_id=WATER_ID, lang=LANG)
meta


                    INFO     get_pictogram_metadata(id=2248): OK from local dataset.                 ]8;id=2645382;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=2645383;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#246\246]8;;\

{'id': 2248,
 'image_url': '/api/images/2248',
 'keywords': [{'type': 2,
   'keyword': 'water',
   'plural': 'waters',
   'meaning': None}],
 'categories': ['beverage', 'mineral rich food'],
 'synsets': ['07951744-n', '14869913-n'],
 'tags': ['feeding', 'food', 'beverage', 'mineral rich food'],
 'sex': False,
 'violence': False,
 'schematic': False,
 'aac': False,
 'aac_color': False,
 'skin': False,
 'hair': False,
 'created': '2007-12-12T10:05:23.000Z',
 'last_updated': '2020-05-29T14:55:35.099Z'}

In [10]:
# unknown id
get_pictogram_metadata(pictogram_id=999_999_999, lang=LANG)


[07/19/26 15:00:17] INFO     get_pictogram_metadata(id=999999999): not found on API.                 ]8;id=2645389;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=2645390;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#260\260]8;;\

{'error': 'Pictogram id=999999999 not found on ARASAAC'}

In [11]:
# roundtrip: first search result --> metadata
first_id = pics[0]['id']
get_pictogram_metadata(pictogram_id=first_id, lang=LANG)


                    INFO     get_pictogram_metadata(id=22082): OK from local dataset.                ]8;id=2645395;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=2645396;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#246\246]8;;\

{'id': 22082,
 'image_url': '/api/images/22082',
 'keywords': [{'type': 3,
   'keyword': 'water',
   'plural': None,
   'meaning': 'tr. Sprinkle water on a surface: watering the garden.'},
  {'type': 3, 'keyword': 'irrigate', 'plural': None, 'meaning': None}],
 'categories': ['verb', 'agriculture', 'gardening'],
 'synsets': ['00228662-v'],
 'tags': ['communication',
  'language',
  'verb',
  'work',
  'primary sector',
  'agriculture',
  'gardening'],
 'sex': False,
 'violence': False,
 'schematic': False,
 'aac': False,
 'aac_color': False,
 'skin': True,
 'hair': True,
 'created': '2010-10-03T17:05:18.000Z',
 'last_updated': '2020-11-21T00:09:27.943Z'}

## 4. `search_pictograms_by_synset`

In [12]:
WATER_SYNSET = '07951744-n'
syn_pics = search_pictograms_by_synset(synset_id=WATER_SYNSET, lang=LANG)['results']

print(f'search_pictograms_by_synset("{WATER_SYNSET}") --> {len(syn_pics)} results')
for p in syn_pics:
    print(f'  id={p["id"]:<8} kws={[k["keyword"] for k in p.get("keywords", [])[:2]]}')


                    INFO     Loaded synset_index for lang='en' (8422 synsets).                 ]8;id=2645402;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/dataset_cache.py\dataset_cache.py]8;;\:]8;id=2645403;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/dataset_cache.py#101\101]8;;\

                    INFO     search_pictograms_by_synset(synset=07951744-n, lang=en): 3 results from ]8;id=2645409;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=2645410;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#325\325]8;;\
                             local dataset.                                                                        

search_pictograms_by_synset("07951744-n") --> 3 results
  id=2248     kws=['water']
  id=32464    kws=['water']
  id=6889     kws=['water']


In [13]:
# unknown synset
print('unknown synset:', search_pictograms_by_synset(synset_id='00000000-x', lang=LANG)['results'])


                    INFO     search_pictograms_by_synset(synset=00000000-x, lang=en): 0 results from ]8;id=2645415;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=2645416;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#325\325]8;;\
                             local dataset.                                                                        

unknown synset: []


In [14]:
# cross-check: a pictogram found via keyword search, looked up again through its own synset
source_pic = next((p for p in pics if p.get('synsets')), None)
if source_pic:
    back = search_pictograms_by_synset(synset_id=source_pic['synsets'][0], lang=LANG)['results']
    back_ids = [p['id'] for p in back]
    print(f'id={source_pic["id"]} looked up via synset {source_pic["synsets"][0]!r} --> found again: {source_pic["id"] in back_ids}')
else:
    print('no synset-bearing pictogram among the results — skipping this check')


                    INFO     search_pictograms_by_synset(synset=00228662-v, lang=en): 6 results from ]8;id=2645421;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=2645422;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#325\325]8;;\
                             local dataset.                                                                        

id=22082 looked up via synset '00228662-v' --> found again: True


## 5. `get_time`

Real system clock (no `MOCK_TIME_INFO` set).

In [15]:
from datetime import datetime

time_result = get_time()
time_result


{'current_dt': '2026-07-19T15:00:17.113346', 'time_of_day': 'afternoon'}

In [16]:
# just eyeballing that the format is sane
print('parses as ISO 8601:', datetime.fromisoformat(str(time_result['current_dt'])))
print('time_of_day:', time_result['time_of_day'])


parses as ISO 8601: 2026-07-19 15:00:17.113346
time_of_day: afternoon


## 6. `get_schedule`

Real calendar provider (no `MOCK_SCHEDULE_EVENTS` set).

In [17]:
import mcp_server.tools.schedule_tool as _schedule_mod


for provider in ('google', 'apple'):
    _schedule_mod.CALENDAR_PROVIDER = provider
    print(f'--- provider={provider} ---')
    try:
        schedule = get_schedule()
        print(f'{len(schedule)} events')
        display(schedule)
    except Exception as exc:
        print(f'FAILED: {type(exc).__name__}: {exc}')


--- provider=google ---


                    INFO     file_cache is only supported with oauth2client<4.0.0                    ]8;id=2645429;file:///Users/pelle/python_venvs/BDATM/lib/python3.11/site-packages/googleapiclient/discovery_cache/__init__.py\__init__.py]8;;\:]8;id=2645430;file:///Users/pelle/python_venvs/BDATM/lib/python3.11/site-packages/googleapiclient/discovery_cache/__init__.py#49\49]8;;\

                    INFO     Google Calendar: 1 events for 2026-07-19                          ]8;id=2645437;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/schedule_tool.py\schedule_tool.py]8;;\:]8;id=2645438;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/schedule_tool.py#145\145]8;;\

1 events


[{'title': 'BDATM test from google calendar',
  'start_time': '16:00:00',
  'location': None,
  'description': None}]

--- provider=apple ---


[07/19/26 15:00:20] WARNING  Retrying (Retry(total=0, connect=None, read=False,              ]8;id=2645445;file:///Users/pelle/python_venvs/BDATM/lib/python3.11/site-packages/urllib3/connectionpool.py\connectionpool.py]8;;\:]8;id=2645446;file:///Users/pelle/python_venvs/BDATM/lib/python3.11/site-packages/urllib3/connectionpool.py#1878\1878]8;;\
                             redirect=None, status=None)) after connection broken by                               
                             'MustDowngradeError('The server yielded its support for                               
                             HttpVersion.h3 through the Alt-Svc header while unable to do                          
                             so. To remediate that issue, either disable HttpVersion.h3 or                         
                             reach out to the server admin.')': /                                                  

[07/19/26 15:00:23] INFO     Apple CalDAV: 1 events for 2026-07-19                             ]8;id=2645452;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/schedule_tool.py\schedule_tool.py]8;;\:]8;id=2645453;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/schedule_tool.py#203\203]8;;\

1 events


[{'title': 'BDATM test from apple calendar',
  'start_time': '15:00:00',
  'location': None,
  'description': None}]

## 7. Online / offline comparison

Compares local dataset output against the live API by monkey-patching
`USE_LOCAL_DATASETS` in the tool module. Requires network: if the connection
is missing, the cell just says so and stops there.


In [18]:
import contextlib

@contextlib.contextmanager
def _use_api():
    old = _arasaac_mod.USE_LOCAL_DATASETS
    _arasaac_mod.USE_LOCAL_DATASETS = False
    try:
        yield
    finally:
        _arasaac_mod.USE_LOCAL_DATASETS = old


In [19]:
try:
    local_kws = set(list_keywords(lang=LANG)['keywords'])
    with _use_api():
        api_kws = set(list_keywords(lang=LANG)['keywords'])
    jaccard = len(local_kws & api_kws) / len(local_kws | api_kws)
    print(f'local={len(local_kws):,}  api={len(api_kws):,}  jaccard={jaccard:.3f}')
    print('local-only, sample:', list(local_kws - api_kws)[:5])
except (ConnectionError, TimeoutError) as exc:
    print(f'SKIP (no network): {exc}')


                    INFO     list_keywords(lang='en'): 15718 keywords from local dataset.            ]8;id=2645458;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=2645459;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#280\280]8;;\

                    INFO     list_keywords(lang='en'): local dataset unavailable, calling API.       ]8;id=2645465;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=2645466;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#286\286]8;;\

                    INFO     list_keywords(lang='en'): 16971 keywords from API.                      ]8;id=2645472;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=2645473;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#299\299]8;;\

local=15,718  api=16,969  jaccard=0.800
local-only, sample: ['ebro river', 'person with ppe', 'united states of america', 'india', 'i spy']


In [20]:
try:
    local_ids = {p['id'] for p in search_pictograms(keyword='water', lang=LANG, max_results=10)['results']}
    with _use_api():
        api_ids = {p['id'] for p in search_pictograms(keyword='water', lang=LANG, max_results=10)['results']}
    print(f'local={sorted(local_ids)}')
    print(f'api={sorted(api_ids)}')
    print(f'common: {sorted(local_ids & api_ids)}')
except (ConnectionError, TimeoutError) as exc:
    print(f'SKIP (no network): {exc}')


                    INFO     search_pictograms('water', lang='en'): 8 results from local dataset.    ]8;id=2645478;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=2645479;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#206\206]8;;\

[07/19/26 15:00:24] INFO     search_pictograms('water', lang='en'): 10 results from API.             ]8;id=2645485;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=2645486;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#219\219]8;;\

local=[2248, 2816, 6889, 22082, 24354, 24355, 32464, 34173]
api=[2248, 2816, 3172, 3277, 6889, 22082, 24354, 24355, 32464, 34173]
common: [2248, 2816, 6889, 22082, 24354, 24355, 32464, 34173]


In [21]:
try:
    local_meta = get_pictogram_metadata(pictogram_id=WATER_ID, lang=LANG)
    with _use_api():
        api_meta = get_pictogram_metadata(pictogram_id=WATER_ID, lang=LANG)
    print('local id:', local_meta['id'], ' api id:', api_meta['id'])
    print('local keywords:', [k['keyword'] for k in local_meta['keywords'][:5]])
    print('api keywords:  ', [k['keyword'] for k in api_meta['keywords'][:5]])
except (ConnectionError, TimeoutError) as exc:
    print(f'SKIP (no network): {exc}')


                    INFO     get_pictogram_metadata(id=2248): OK from local dataset.                 ]8;id=2645491;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=2645492;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#246\246]8;;\

                    INFO     get_pictogram_metadata(id=2248): OK from API.                           ]8;id=2645498;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=2645499;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#264\264]8;;\

local id: 2248  api id: 2248
local keywords: ['water']
api keywords:   ['water']


In [22]:
try:
    local_syn = {p['id'] for p in search_pictograms_by_synset(synset_id=WATER_SYNSET, lang=LANG)['results']}
    with _use_api():
        api_syn = {p['id'] for p in search_pictograms_by_synset(synset_id=WATER_SYNSET, lang=LANG)['results']}
    print(f'local={sorted(local_syn)}  api={sorted(api_syn)}')
except (ConnectionError, TimeoutError) as exc:
    print(f'SKIP (no network): {exc}')


                    INFO     search_pictograms_by_synset(synset=07951744-n, lang=en): 3 results from ]8;id=2645504;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=2645505;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#325\325]8;;\
                             local dataset.                                                                        

                    INFO     search_pictograms_by_synset(synset=07951744-n, wn=3.1, lang=en): 3      ]8;id=2645511;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=2645512;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#350\350]8;;\
                             results from API.                                                                     

local=[2248, 6889, 32464]  api=[2248, 6889, 32464]


## 8. `resolve_concept`

In [23]:
from agent.resolve import resolve_concept

try:
    from agent.session import _nlp
    _nlp()
    _SPACY_OK = True
except Exception as exc:
    _SPACY_OK = False
    print(f'spaCy not available: {exc}')

print('spaCy available:', _SPACY_OK)

spaCy available: True


In [24]:
# exact match
print(resolve_concept('water', kw_set))
print(resolve_concept('eat', kw_set))


['water']
['eat']


In [25]:
# whole-phrase lemma (requires spaCy)
if _SPACY_OK and 'go out' in kw_set:
    print(resolve_concept('going out', kw_set))
else:
    print('skipping, spaCy unavailable or "go out" not in the index')


['go out']


In [26]:
# space <-> hyphen normalisation: find a real case the index actually supports
for kw in sorted(kw_set):
    if '-' in kw:
        space_form = kw.replace('-', ' ')
        if space_form not in kw_set and kw in resolve_concept(space_form, kw_set):
            print(f'{space_form!r} ==> {resolve_concept(space_form, kw_set)}')
            break
else:
    print('no hyphen/space case found in the index')


'1 2 3' ==> ['1-2-3']


In [27]:
# token fallback, for a multi-word phrase not present in the index
print(resolve_concept('wash hands', kw_set))


['wash', 'hand']


In [28]:
# edge cases
print('empty:', resolve_concept('', kw_set))
print('unknown:', resolve_concept('jbfeifheuhfk', kw_set))


empty: []


[07/19/26 15:00:38] INFO     Loaded keyword embeddings for lang='en': 15718 keywords, dim=384.        ]8;id=2645519;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/agent/resolve.py\resolve.py]8;;\:]8;id=2645520;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/agent/resolve.py#65\65]8;;\

/Users/pelle/python_venvs/BDATM/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[07/19/26 15:00:42] INFO     Use pytorch device_name: cpu                                ]8;id=2645527;file:///Users/pelle/python_venvs/BDATM/lib/python3.11/site-packages/sentence_transformers/SentenceTransformer.py\SentenceTransformer.py]8;;\:]8;id=2645528;file:///Users/pelle/python_venvs/BDATM/lib/python3.11/site-packages/sentence_transformers/SentenceTransformer.py#210\210]8;;\

                    INFO     Load pretrained SentenceTransformer: all-MiniLM-L6-v2       ]8;id=2645534;file:///Users/pelle/python_venvs/BDATM/lib/python3.11/site-packages/sentence_transformers/SentenceTransformer.py\SentenceTransformer.py]8;;\:]8;id=2645535;file:///Users/pelle/python_venvs/BDATM/lib/python3.11/site-packages/sentence_transformers/SentenceTransformer.py#218\218]8;;\

[07/19/26 15:00:43] WARNING  Retrying (Retry(total=0, connect=None, read=False,              ]8;id=2645540;file:///Users/pelle/python_venvs/BDATM/lib/python3.11/site-packages/urllib3/connectionpool.py\connectionpool.py]8;;\:]8;id=2645541;file:///Users/pelle/python_venvs/BDATM/lib/python3.11/site-packages/urllib3/connectionpool.py#1878\1878]8;;\
                             redirect=None, status=None)) after connection broken by                               
                             'MustDowngradeError('The server yielded its support for                               
                             HttpVersion.h3 through the Alt-Svc header while unable to do                          
                             so. To remediate that issue, either disable HttpVersion.h3 or                         
                             reach out to the server admin.')':                                                    
                             /api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v                       
                             2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json                               

[07/19/26 15:00:46] INFO     Loaded SentenceTransformer model 'all-MiniLM-L6-v2' for embedding        ]8;id=2645547;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/agent/resolve.py\resolve.py]8;;\:]8;id=2645548;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/agent/resolve.py#86\86]8;;\
                             lookup.                                                                               

unknown: []


In [34]:
# return_method=True, also see which method was used to resolve
print(resolve_concept('water', kw_set, return_method=True))
print(resolve_concept('going', kw_set, return_method=True))


(['water'], 'exact')
(['go'], 'lemma')
